# 03 - GRPO with DIS

This notebook keeps GRPO's eight-solution group advantage but removes PPO clipping. For each generated token, `ratio = current policy / rollout policy`. Tokens outside the strict interval `0.2 < ratio < 4.0` are removed from the policy gradient entirely; ratios inside the interval are not capped.

The actor and frozen reference load the private hosted SFT checkpoint with `HF_TOKEN`; rerunning `00_sft.ipynb` is not required.

In [1]:
# Hyperparameters
sft_checkpoint = "ppbhatt500/rl-ablations-sft-2026-07-17"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 256
eval_problems = 64
num_epochs = 1
group_size = 8
unique_prompts_per_update = 16
completions_per_update = unique_prompts_per_update * group_size
train_micro_batch_size = 8
gradient_accumulation_steps = 16
generation_batch_size = 128
generation_micro_batch_size = 16
max_prompt_tokens = 384
max_completion_tokens = 512
think_tag_reward_weight = 0.25
length_reward_weight = 0.25
length_reward_target_tokens = 512
temperature = 0.8
top_p = 1.0
learning_rate = 1e-6
weight_decay = 0.0
epsilon_low = 0.8
epsilon_high = 3.0
max_grad_norm = 1.0
eval_every_steps = 4
eval_batch_size = 16
wandb_project = "swe-rl-ablations"
wandb_run_name = "grpo-dis-2"
output_path = "./checkpoints/grpo-dis-2"

optimizer_steps = train_problems // unique_prompts_per_update
assert completions_per_update == 128
assert train_micro_batch_size * gradient_accumulation_steps == completions_per_update
assert max_completion_tokens == length_reward_target_tokens == 512
assert think_tag_reward_weight == length_reward_weight == 0.25
assert epsilon_low == 0.8 and epsilon_high == 3.0
assert train_problems == 256 and eval_problems == 64 and num_epochs == 1
print(f"DIS keeps tokens only when {1-epsilon_low} < ratio < {1+epsilon_high}")

DIS keeps tokens only when 0.19999999999999996 < ratio < 4.0


In [2]:
from pathlib import Path
import os
import re
import time

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import GRPOConfig, GRPOTrainer

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = sft_checkpoint
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is required to load the private SFT checkpoint.")
os.environ["WANDB_PROJECT"] = wandb_project
set_seed(seed)

train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision).shuffle(seed=seed).select(range(train_problems))
eval_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision).shuffle(seed=seed).select(range(eval_problems))

In [3]:
def gold_number(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K reference answer: {answer!r}")
    return match.group(1).replace(",", "")


def completion_text(completion):
    return completion[-1]["content"] if isinstance(completion, list) else completion


def predicted_number(completion):
    completion = completion_text(completion)
    match = re.search(r"Final answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", completion.strip(), re.IGNORECASE)
    return None if not match else match.group(1).replace(",", "")


def exact_answer_reward(completions, answer, **_):
    rewards = []
    for completion, reference in zip(completions, answer):
        prediction = predicted_number(completion)
        rewards.append(-1.0 if prediction is None else float(float(prediction) == float(gold_number(reference))))
    return rewards


def think_tag_reward(completions, **_):
    return [
        think_tag_reward_weight
        if re.search(r"<think>\s*\S[\s\S]*?</think>", completion_text(completion), re.IGNORECASE)
        else 0.0
        for completion in completions
    ]


def length_reward(completions, **_):
    texts = [completion_text(completion) for completion in completions]
    token_counts = [
        len(input_ids)
        for input_ids in tokenizer(texts, add_special_tokens=False, truncation=False)["input_ids"]
    ]
    return [
        length_reward_weight * min(count / length_reward_target_tokens, 1.0)
        for count in token_counts
    ]


def make_prompt(question):
    return [
        {
            "role": "system",
            "content": (
                "Think carefully and reason step by step before answering. Put your full "
                "reasoning inside <think>...</think>, then give `Final answer: <number>` "
                "after the closing </think> tag."
            ),
        },
        {
            "role": "user",
            "content": (
                "Solve this grade-school math problem. Work through it step by step before "
                f"answering.\n\n{question}"
            ),
        },
    ]

train_rows = train_rows.map(lambda row: {"prompt": make_prompt(row["question"])})
eval_rows = eval_rows.map(lambda row: {"prompt": make_prompt(row["question"])})

tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=True, token=hf_token)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

actor = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2", token=hf_token)
reference = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2", token=hf_token).eval().to("cuda")
for parameter in reference.parameters():
    parameter.requires_grad_(False)

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

TRL still handles prompt grouping, rollouts, reward standardization, buffering, optimization, and checkpoints. The overridden loss is the method ablation: standard clipping is replaced by the strict DIS mask.

In [4]:
class DISGRPOTrainer(GRPOTrainer):
    def __init__(self, *args, telemetry_reference, generation_micro_batch_size, exact_eval_rows, epsilon_low, epsilon_high, **kwargs):
        super().__init__(*args, **kwargs)
        self.telemetry_reference = telemetry_reference
        self.generation_micro_batch_size = generation_micro_batch_size
        self.exact_eval_rows = exact_eval_rows
        self.dis_epsilon_low = epsilon_low
        self.dis_epsilon_high = epsilon_high

    def _generate_single_turn(self, prompt_ids, images, multimodal_fields):
        completion_ids = []
        logprobs = []
        has_logprobs = True
        for start in range(0, len(prompt_ids), self.generation_micro_batch_size):
            chunk_ids, chunk_logprobs = super()._generate_single_turn(
                prompt_ids[start:start + self.generation_micro_batch_size],
                None if images is None else images[start:start + self.generation_micro_batch_size],
                multimodal_fields,
            )
            completion_ids.extend(chunk_ids)
            if chunk_logprobs is None:
                has_logprobs = False
            else:
                logprobs.extend(chunk_logprobs)
        return completion_ids, logprobs if has_logprobs else None

    def _compute_loss(self, model, inputs):
        prompt_ids = inputs["prompt_ids"]
        prompt_mask = inputs["prompt_mask"]
        completion_ids = inputs["completion_ids"]
        completion_mask = inputs["completion_mask"].float()
        input_ids = torch.cat([prompt_ids, completion_ids], dim=1)
        attention_mask = torch.cat([prompt_mask, inputs["completion_mask"]], dim=1)

        current_logps, _, _ = self._get_per_token_logps_and_entropies(
            model, input_ids, attention_mask, completion_ids.size(1), compute_entropy=False
        )
        rollout_logps = inputs.get("old_per_token_logps")
        if rollout_logps is None:
            rollout_logps = current_logps.detach()
        advantages = inputs["advantages"]
        if advantages.ndim == 1:
            advantages = advantages.unsqueeze(1)

        ratio = torch.exp(current_logps - rollout_logps)
        active = ((ratio > 1 - self.dis_epsilon_low) & (ratio < 1 + self.dis_epsilon_high)).float()
        token_loss = -ratio * advantages * active
        sequence_loss = (token_loss * completion_mask).sum(1) / completion_mask.sum(1).clamp_min(1)
        loss = sequence_loss.mean() / self.current_gradient_accumulation_steps

        with torch.no_grad():
            reference_logits = self.telemetry_reference(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits[:, -completion_ids.size(1)-1:-1]
            reference_logps = torch.log_softmax(reference_logits.float(), dim=-1).gather(-1, completion_ids.unsqueeze(-1)).squeeze(-1)
            kl = ((current_logps - reference_logps) * completion_mask).sum() / completion_mask.sum().clamp_min(1)
            masked_fraction = ((1 - active) * completion_mask).sum() / completion_mask.sum().clamp_min(1)
            self._metrics["train"]["kl_sft"].append(float(kl.item()))
            self._metrics["train"]["dis_masked_fraction"].append(float(masked_fraction.item()))
        return loss

    def training_step(self, *args, **kwargs):
        torch.cuda.synchronize()
        started_at = time.perf_counter()
        loss = super().training_step(*args, **kwargs)
        torch.cuda.synchronize()
        self._last_policy_step_seconds = time.perf_counter() - started_at
        return loss

    def log(self, logs, *args, **kwargs):
        logs = dict(logs)
        if "reward" in logs:
            logs["train/reward"] = logs["reward"]
        if "loss" in logs:
            logs["loss/policy"] = logs["loss"]
        if "grad_norm" in logs:
            logs["grad_norm/policy"] = logs["grad_norm"]
        if "kl_sft" in logs:
            logs["objective/kl_sft"] = logs["kl_sft"]
        if "dis_masked_fraction" in logs:
            logs["policy/tokens_clipped_or_masked_pct"] = 100 * logs["dis_masked_fraction"]
        if hasattr(self, "_last_policy_step_seconds"):
            logs["time/policy_forward_backward_s"] = self._last_policy_step_seconds
            logs["time/step_s"] = self._last_policy_step_seconds
        logs["gpu/peak_memory_gb"] = torch.cuda.max_memory_allocated() / 1024**3
        return super().log(logs, *args, **kwargs)

    @torch.no_grad()
    def evaluate(self, *args, **kwargs):
        model = self.accelerator.unwrap_model(self.model)
        model.eval()
        rewards = []
        for start in range(0, len(self.exact_eval_rows), eval_batch_size):
            stop = min(start + eval_batch_size, len(self.exact_eval_rows))
            rows = self.exact_eval_rows.select(range(start, stop))
            prompts = [tokenizer.apply_chat_template(make_prompt(question), tokenize=False, add_generation_prompt=True) for question in rows["question"]]
            encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_prompt_tokens).to(model.device)
            generated = model.generate(**encoded, do_sample=False, max_new_tokens=max_completion_tokens, pad_token_id=tokenizer.pad_token_id)
            completions = tokenizer.batch_decode(generated[:, encoded.input_ids.shape[1]:], skip_special_tokens=True)
            rewards.extend(exact_answer_reward(completions, rows["answer"]))
        model.train()
        score = sum(rewards) / len(rewards)
        self.log({"eval/reward": score})
        return {"eval/reward": score}


In [5]:
# TRL derives the intended 128-completion batch from 8 examples x 16 steps.
# Passing both controls is rejected, so leave the derived field unset.
generation_batch_size = None

In [6]:
import wandb

training_args = GRPOConfig(
    output_dir=str(project_root / output_path),
    run_name=wandb_run_name,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    generation_batch_size=generation_batch_size,
    steps_per_generation=gradient_accumulation_steps,
    num_generations=group_size,
    num_generations_eval=1,
    max_completion_length=max_completion_tokens,
    temperature=temperature,
    top_p=top_p,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    max_grad_norm=max_grad_norm,
    beta=0.0,
    scale_rewards="group",
    loss_type="grpo",
    importance_sampling_level="token",
    eval_strategy="steps",
    eval_steps=eval_every_steps,
    logging_steps=1,
    save_strategy="epoch",
    report_to="wandb",
    bf16=True,
    gradient_checkpointing=True,
)

trainer = DISGRPOTrainer(
    model=actor,
    telemetry_reference=reference,
    generation_micro_batch_size=generation_micro_batch_size,
    exact_eval_rows=eval_rows,
    epsilon_low=epsilon_low,
    epsilon_high=epsilon_high,
    reward_funcs=[exact_answer_reward, think_tag_reward, length_reward],
    args=training_args,
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    processing_class=tokenizer,
)

trl_log = trainer.log
def log_comparable_metrics(logs, *args, **kwargs):
    if "eval/reward" in logs:
        if wandb.run is not None:
            wandb.log({"eval/reward": logs["eval/reward"]}, step=trainer.state.global_step)
        return
    trl_log(logs, *args, **kwargs)
    comparable = {}
    if "reward" in logs:
        comparable["train/reward"] = logs["reward"]
    if "kl_sft" in logs:
        comparable["objective/kl_sft"] = logs["kl_sft"]
    if "dis_masked_fraction" in logs:
        comparable["policy/tokens_clipped_or_masked_pct"] = 100 * logs["dis_masked_fraction"]
    for key in ("loss/policy", "grad_norm/policy", "time/step_s", "time/policy_forward_backward_s", "gpu/peak_memory_gb"):
        if key in logs:
            comparable[key] = logs[key]
    if comparable and wandb.run is not None:
        wandb.log(comparable, step=trainer.state.global_step)
trainer.log = log_comparable_metrics

trainer.train()
trainer.save_model(str(project_root / output_path))
tokenizer.save_pretrained(str(project_root / output_path))

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: ppbhatt500 (ppbhatt500-verizon) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /home/ubuntu/rl-ablations/notebooks/wandb/run-20260721_234855-mfydb7cg
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run grpo-dis-2


...

wandb: WARNING Tried to log to step 16 that is less than the current step 976. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


wandb: WARNING Tried to log to step 16 that is less than the current step 976. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/ubuntu/rl-ablations/checkpoints/grpo-dis-2/tokenizer_config.json',
 '/home/ubuntu/rl-ablations/checkpoints/grpo-dis-2/chat_template.jinja',
 '/home/ubuntu/rl-ablations/checkpoints/grpo-dis-2/tokenizer.json')